# StatD2GAN v2 — Colab runner

Thin orchestration notebook. All model/training/analysis logic lives in the `statd2gan` package under `src/` — this notebook only wires it to Google Drive and Colab's GPU runtime.

Expected data layout on Drive: `MyDrive/statd2gan_paper/{ankara,yakutsk,lhasa,dubai,bergen}.csv` (raw ERA5 exports).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/<your-username>/statd2gan.git /content/statd2gan
%pip install -q -e /content/statd2gan

In [ ]:
import os
os.environ['STATD2GAN_DATA_ROOT'] = '/content/drive/MyDrive/statd2gan_paper'

from statd2gan.config import C, DEVICE, LOCATIONS, make_paths
from statd2gan.preprocessing import preprocess_location
from statd2gan.data import load_location
from statd2gan.ablation import ABLATION_CONFIGS, run_ablation, run_sensitivity, calibration_ablation
from statd2gan.plotting import plot_ablation, plot_sensitivity

print('Device:', DEVICE)

## Run one location

Resumable: re-running this cell after a Colab disconnect skips every (config, seed) already checkpointed.

In [ ]:
LOCATION = 'bergen'  # <- pick a location

paths = make_paths(LOCATION, C.base_root)
assert preprocess_location(LOCATION, paths, C.base_root, C), f'{LOCATION}.csv not found under {C.base_root}'
loc_data = load_location(paths, C)

df_abl = run_ablation(DEVICE, paths, loc_data.train_tensor, loc_data.real_train_phys,
                       loc_data.real_val_phys, loc_data.scaler)
plot_ablation(paths, df_abl, n_seeds=len(C.seeds))

In [ ]:
df_sens = run_sensitivity(DEVICE, paths, loc_data.train_tensor, loc_data.real_train_phys,
                           loc_data.real_val_phys, loc_data.scaler)
plot_sensitivity(paths, df_sens)
calibration_ablation(paths, df_abl)

## Cross-location analysis

Run only after every location above has a complete `ablation_raw.csv`. No training happens here.

In [ ]:
from statd2gan.analysis import (backfill_diagnostics, load_all, make_summary,
                                 pct_table, wilcoxon_table, write_latex)

cfg_by_name = {c.name: c for c in ABLATION_CONFIGS}
contrasts = [n for n in cfg_by_name if n != 'full_model']
OUT_DIR = f'{C.base_root}/final_analysis'
os.makedirs(OUT_DIR, exist_ok=True)

backfill_diagnostics(DEVICE, cfg_by_name, C.base_root)
df, floors = load_all(C.base_root, out_dir=OUT_DIR)
summary = make_summary(df, floors, OUT_DIR)
pct = pct_table(df, contrasts, OUT_DIR)
wtab = wilcoxon_table(df, contrasts, OUT_DIR)
write_latex(summary, wtab, floors, contrasts, OUT_DIR)
wtab